In [ ]:
# Download folder data dan src (uncomment jika di Colab)
# !svn checkout https://github.com/teranixbq/IR-BERT-Transformer/trunk/data
# !svn checkout https://github.com/teranixbq/IR-BERT-Transformer/trunk/src

# Neural IR Exercise dengan BERT Cross-Encoder
Dosen: Zico Pratama Putra
Kelompok: [Nama 1] - [Nama 2] - [Nama 3]
Tanggal: ---

In [ ]:
from src.judgement_aggregation import load_raw_judgements, aggregate_judgements, save_qrels, compute_annotator_reliability
from src.bert_cross_encoder import BERTCrossEncoder
from transformers import pipeline
import pandas as pd

In [ ]:
df = load_raw_judgements("data/fira_raw_judgements.tsv")

agg_maj = aggregate_judgements(df, method='majority')
agg_weighted = aggregate_judgements(df, method='advanced', strategy='confidence_weighted')

reliability = compute_annotator_reliability(df)
agg_reliability = aggregate_judgements(df, method='advanced', strategy='annotator_reliability', annotator_reliability=reliability)

save_qrels(agg_maj, "data/fira_aggregated.qrels")
save_qrels(agg_weighted, "data/fira_aggregated_confidence_weighted.qrels")
save_qrels(agg_reliability, "data/fira_aggregated_annotator_reliability.qrels")

print("=== PERBANDINGAN BASELINE (MAJORITY) VS ADVANCED ===")
print(f"{'Method':<30} {'Score Dist':<30} {'Mean Std Score':<15}")
print("-" * 75)
for name, result in [('majority (baseline)', agg_maj), 
                       ('confidence_weighted', agg_weighted), 
                       ('annotator_reliability', agg_reliability)]:
    dist = result['score'].value_counts().sort_index().to_dict()
    mean_std = result['std_score'].mean()
    print(f"{name:<30} {str(dist):<30} {mean_std:<15.3f}")

print("\n=== ANALISIS MANUAL: Contoh Query-Doc Pair ===")
sample_pairs = [(1, 'd1_3'), (1, 'd1_8'), (2, 'd2_1')]
for qid, did in sample_pairs:
    raw = df[(df['query_id'] == qid) & (df['doc_id'] == did)]
    print(f"\nQuery {qid}, Doc {did}:")
    for _, row in raw.iterrows():
        rel = reliability.get(row['annotator_id'], 0)
        print(f"  {row['annotator_id']}: judgement={row['judgement']}, confidence={row['confidence']:.3f}, reliability={rel:.3f}")
    maj_score = agg_maj[(agg_maj['query_id'] == qid) & (agg_maj['doc_id'] == did)]['score'].values[0]
    w_score = agg_weighted[(agg_weighted['query_id'] == qid) & (agg_weighted['doc_id'] == did)]['score'].values[0]
    r_score = agg_reliability[(agg_reliability['query_id'] == qid) & (agg_reliability['doc_id'] == did)]['score'].values[0]
    print(f"  -> majority={maj_score}, confidence_weighted={w_score}, annotator_reliability={r_score}")

## Part 2: BERT Cross-Encoder Re-Ranking

In [ ]:
reranker = BERTCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

query = "How to make a good cappuccino?"
passages = ["Passage 1 ...", "Passage 2 ...", "Passage 3 ..."]
ranked_idx, scores = reranker.re_rank(query, passages)
print("Ranked order:", ranked_idx)

## Part 3: Extractive QA

In [ ]:
qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")

result = qa_pipeline(question=query, context=passages[ranked_idx[0]])
print(result)

## Evaluasi
# TODO: Implement evaluasi MRR@10, NDCG@10, dll.